# Multi-Agent · Day 40 — **Semantic Memory & Vector Store Integration**

**Phase 2 · Module 6: Agent Memory, Tools & MCP** · ~2.5 hrs

Episodic memory (Day 39) answered "what happened with *this* customer?" — scoped, recency-weighted. **Semantic memory** is the other axis: durable, *customer-agnostic* knowledge the agent has learned — insights, patterns, distilled facts — retrieved by meaning, shared across every session and every customer.

"SME manufacturers in the North-West with arrears tend to recover within two quarters" is semantic: no single customer owns it, and you want it surfaced whenever it's relevant. Today we store agent **insights** as embeddings, retrieve by similarity, add **consolidation** (so memory stays compact), and pick the right vector store for the job.

Real FAISS; same deterministic offline embedder as Day 39 (swap in `sentence-transformers` for real semantics).

Today:
1. **Semantic memory** — store insights as embeddings, retrieve by meaning.
2. **Global scope** — why semantic memory is *not* filtered by customer.
3. **Consolidation** — summarise old memories so retrieval stays sharp.
4. **FAISS vs Pinecone vs OpenSearch** — the store-selection decision.

## 1. Semantic memory — insights as embeddings

An agent finishes a task and writes down what it *learned*, not what happened. Store the insight text as a vector; retrieve later by semantic similarity to the current situation.

In [1]:
import numpy as np, hashlib, faiss, time
from dataclasses import dataclass, field

DIM = 64
def _bucket(tok): return int(hashlib.md5(tok.encode()).hexdigest(), 16) % DIM
def encode(texts):
    v = np.zeros((len(texts), DIM), "float32")
    for i, t in enumerate(texts):
        for tok in t.lower().split():
            v[i, _bucket(tok)] += 1.0
    return v / np.clip(np.linalg.norm(v, axis=1, keepdims=True), 1e-9, None)

@dataclass
class Insight:
    text: str
    source: str                       # which agent/run learned it
    hits: int = 0                     # times retrieved -> usefulness signal
    ts: float = field(default_factory=time.time)

class SemanticMemory:
    def __init__(self, dim=DIM):
        self.index = faiss.IndexFlatIP(dim)
        self.items: list[Insight] = []
    def store(self, insight: Insight):
        self.index.add(encode([insight.text])); self.items.append(insight)
    def retrieve(self, query: str, top_k=3):
        if not self.items: return []
        scores, idxs = self.index.search(encode([query]), min(top_k, len(self.items)))
        out = []
        for j, i in enumerate(idxs[0]):
            if i == -1: continue
            self.items[i].hits += 1          # track usefulness
            out.append((float(scores[0][j]), self.items[i].text))
        return out

sem = SemanticMemory()
for txt in ["SME manufacturers with short arrears usually recover within two quarters",
            "sanctions hits on shell companies cluster in high-turnover low-headcount firms",
            "overdraft fee disputes spike after quarterly statement runs",
            "mortgage applications from contractors need two years of accounts"]:
    sem.store(Insight(txt, source="analysis-agent"))

print("recall for 'SME arrears recovery outlook':")
for score, text in sem.retrieve("SME arrears recovery outlook"):
    print(f"  {score:.2f}  {text}")

recall for 'SME arrears recovery outlook':
  0.35  SME manufacturers with short arrears usually recover within two quarters
  0.13  sanctions hits on shell companies cluster in high-turnover low-headcount firms
  0.00  overdraft fee disputes spike after quarterly statement runs


☝ Semantic memory stores **distilled knowledge**, not raw events — the unit is an `Insight` the agent chose to remember. The `hits` counter is doing quiet work: every retrieval marks an insight as useful, giving you a usefulness signal to rank by and, later, to decide what to keep during consolidation. This is the "external semantic" memory from Day 39's taxonomy, made concrete with the same FAISS plumbing as episodic — only the *scope* differs, which is cell 2.

## 2. Global scope — the critical difference from episodic

Episodic memory **must** filter by customer (Day 39) — a compliance boundary. Semantic memory is the opposite: it's deliberately **global**, so a pattern learned on one customer helps decide another. Getting this backwards is a real bug.

In [2]:
# episodic: scoped -> customer A's event is unreachable for customer B (correct)
# semantic: global  -> an insight learned on A SHOULD inform B (correct)

def analyse_customer(customer_id: str, situation: str, sem: SemanticMemory):
    relevant = sem.retrieve(situation, top_k=2)
    return {"customer": customer_id,
            "applied_insights": [t for _, t in relevant if _ > 0]}

# same insight surfaces for two DIFFERENT customers — that's the point
print(analyse_customer("C-4417", "SME arrears recovery", sem))
print(analyse_customer("C-8820", "SME arrears recovery", sem))

{'customer': 'C-4417', 'applied_insights': ['SME manufacturers with short arrears usually recover within two quarters', 'sanctions hits on shell companies cluster in high-turnover low-headcount firms']}
{'customer': 'C-8820', 'applied_insights': ['SME manufacturers with short arrears usually recover within two quarters', 'sanctions hits on shell companies cluster in high-turnover low-headcount firms']}


☝ The same insight is applied to C-4417 **and** C-8820 — and that's correct, because it's general knowledge, not either customer's private data. The design rule to state in an interview: **scope by the memory's nature, not by habit.** Episodic = private, per-subject, filtered; semantic = general, shared, unfiltered. The danger is leakage in *either* direction — a customer's raw episode leaking into global semantic memory (privacy breach), or semantic knowledge wrongly scoped-out (agent re-learns the same lesson forever). Keep the two stores separate.

## 3. Consolidation — keep memory sharp

Semantic memory that only grows gets slow and noisy. Consolidation runs periodically: drop never-used insights, and **summarise** clusters of similar ones into a single stronger memory. Here we consolidate near-duplicates by similarity.

In [3]:
def consolidate(sem: SemanticMemory, dup_threshold=0.9, min_hits=0):
    """Merge near-duplicate insights; drop unused ones. Returns a fresh memory."""
    kept: list[Insight] = []
    for item in sorted(sem.items, key=lambda x: x.hits, reverse=True):
        if item.hits < min_hits:                       # prune never-retrieved
            continue
        q = encode([item.text])[0]
        is_dup = any(float(q @ encode([k.text])[0]) >= dup_threshold for k in kept)
        if not is_dup:
            kept.append(item)                          # keep the higher-hit representative
    fresh = SemanticMemory()
    for k in kept: fresh.store(k)
    return fresh

# add a near-duplicate: same insight, one extra word -> high cosine overlap
sem.store(Insight("SME manufacturers with short arrears usually recover within two quarters typically",
                  source="analysis-agent"))
print("before consolidation:", len(sem.items), "insights")
sem2 = consolidate(sem, dup_threshold=0.9)
print("after  consolidation:", len(sem2.items), "insights (near-dup merged)")

before consolidation: 5 insights
after  consolidation: 4 insights (near-dup merged)


☝ Consolidation is what separates a memory system from a landfill. Two levers here: **prune** by the `hits` usefulness signal (an insight never retrieved is dead weight), and **de-duplicate** by similarity so ten paraphrases of one lesson collapse to the strongest representative. Production systems (Mem0, Zep) run this with an LLM that *summarises* a cluster into a new, sharper insight rather than just picking one — but the trigger (similarity + usage) and the goal (compact, high-signal recall) are exactly this. Run it on a schedule, off the hot path.

## 4. FAISS vs Pinecone vs OpenSearch — the store decision

FAISS is perfect for a notebook and small in-process indexes. Production agent memory usually needs persistence, scale, and filtered search — which is where managed stores come in. Pick per requirement.

In [4]:
stores = [
    # (store,      hosting,        best when...,                         watch out)
    ("FAISS",      "in-process lib", "small/medium, single node, fast prototyping", "no persistence/HA — you manage it"),
    ("pgvector",   "Postgres ext",   "already on Postgres; SQL filters + vectors together", "scale limits vs dedicated ANN"),
    ("Pinecone",   "managed SaaS",   "large scale, low-ops, metadata filtering", "cost; data leaves your VPC"),
    ("OpenSearch", "AWS managed",    "already on AWS; hybrid keyword+vector", "operational heft; tuning"),
]
w = max(len(s) for s, *_ in stores)
for s, host, best, watch in stores:
    print(f"{s:{w}}  [{host:14}] {best}\n{'':{w}}   ^ watch: {watch}")

FAISS       [in-process lib] small/medium, single node, fast prototyping
             ^ watch: no persistence/HA — you manage it
pgvector    [Postgres ext  ] already on Postgres; SQL filters + vectors together
             ^ watch: scale limits vs dedicated ANN
Pinecone    [managed SaaS  ] large scale, low-ops, metadata filtering
             ^ watch: cost; data leaves your VPC
OpenSearch  [AWS managed   ] already on AWS; hybrid keyword+vector
             ^ watch: operational heft; tuning


☝ The AVP-level answer isn't "use X" — it's matching the store to constraints. **Already on Postgres?** pgvector keeps vectors next to your relational data and lets one SQL query filter *and* rank (tomorrow, Day 40 Python). **Huge scale, want low-ops?** Pinecone. **On AWS, need hybrid keyword+vector?** OpenSearch. **FAISS** is the right answer for prototypes and embedded indexes — which is why it's the teaching tool, not the production default. Crucially, all four expose the same `store`/`retrieve` shape, so the interface your agents depend on survives the swap.

## 5. Recap

| Concept | Mechanism | Cell |
|---|---|---|
| Insights as vectors | FAISS `IndexFlatIP` + `Insight` metadata | 1 |
| Usefulness signal | `hits` counter per retrieval | 1, 3 |
| Global scope | **no** customer filter (vs episodic) | 2 |
| Keep memory compact | consolidate: prune unused + merge dups | 3 |
| Store selection | FAISS / pgvector / Pinecone / OpenSearch | 4 |

**One-liner:** *episodic is private and scoped; semantic is general and global — both retrieved by similarity, both kept sharp by consolidation.* Tomorrow (Day 41): tool design & schemas — the other half of Module 6, giving agents well-described, versioned tools.

Today's Python notebook builds the **pgvector + asyncpg** path — the same similarity search backed by Postgres instead of FAISS.